# 19/ Customer Segmentation

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

> Mall customer data: CustomerID, Gender, Age, Annual Income (k$), Spending Score (1-100).

**Goal:** Segment mall customers into groups based on income and spending behavior for targeted marketing.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_style("whitegrid")


In [ ]:
df = pd.read_csv("data/Mall_Customers.csv")
print(df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
df.describe().T
sns.pairplot(df[["Annual Income (k$)", "Spending Score (1-100)", "Age"]])
plt.show()


## Preprocessing & Scaling

In [ ]:
X = df[["Annual Income (k$)", "Spending Score (1-100)", "Age"]].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Finding Optimal K (Elbow Method + Silhouette)

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(K_range), inertias, marker='o'); ax[0].set_title("Elbow Method"); ax[0].set_xlabel("k")
ax[1].plot(list(K_range), sil_scores, marker='o', color='orange'); ax[1].set_title("Silhouette Score"); ax[1].set_xlabel("k")
plt.show()


## Final Clustering

In [ ]:
K_FINAL = 5  # pick based on the elbow/silhouette plots above
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df.loc[X.index, 'Cluster'] = kmeans.fit_predict(X_scaled)
df['Cluster'] = df['Cluster'].astype('Int64')
df.head()


## Visualizing Clusters (PCA 2D projection)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(7,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df.loc[X.index, 'Cluster'], palette='tab10')
plt.title("Customer Segments (PCA projection)")
plt.show()


## Cluster Profiles

In [ ]:
df.groupby('Cluster')[["Annual Income (k$)", "Spending Score (1-100)", "Age"]].mean().round(1)


## Next Steps
- Try `DBSCAN` or `Gaussian Mixture Models` for non-spherical clusters.
- Give each segment a business-friendly persona name.
- Feed cluster labels into targeted marketing campaigns.